# Knife blade block: tapered self-centring slot prototype

De-risks the core geometry for the `knife-blade-block` change before writing production code: a single
V-shaped slot, wide at the mouth and narrowing to a constant-width relief channel, that self-centres
knife blades of different spine thickness (2-3 mm for the target Prima set) and keeps the cutting edge
floating clear of the block material. See `openspec/changes/knife-blade-block/design.md` (Decision 3).

## 1. Imports and slot parameters

The slot cross-section (in the blade-thickness / depth-into-block plane) is a symmetric hexagon: a flat
mouth, two tapered walls narrowing to a relief width, then a short constant-width relief channel.

In [1]:
from build123d import (
    MM,
    Align,
    Box,
    BuildLine,
    BuildPart,
    BuildSketch,
    Line,
    Location,
    Mode,
    Plane,
    add,
    extrude,
    make_face,
)

MOUTH_WIDTH = 4.5 * MM  # admits the thickest supported spine (3 mm) with clearance
APEX_WIDTH = 1.0 * MM  # narrower than the thinnest supported spine (2 mm), so it always wedges above this
TAPER_DEPTH = 12.0 * MM  # vertical extent of the sloped part of the V
RELIEF_DEPTH = 3.0 * MM  # constant-width channel below the taper, so the edge never touches solid
DECK_BELOW = 3.0 * MM  # solid material required below the relief for strength
BLOCK_HEIGHT = TAPER_DEPTH + RELIEF_DEPTH + DECK_BELOW
LANE_LENGTH = 84.0 * MM  # 2 Gridfinity units
BLOCK_WIDTH = 40.0 * MM  # arbitrary width for a single-slot prototype

## 2. Build one slot in a test block

**Plane-orientation note:** build123d's `Plane(origin, x_dir, z_dir)` derives the local Y axis as
`cross(z_dir, x_dir)`. To get a sketch whose local X is the world X (blade-thickness direction) and
whose local Y is *downward* into the block (world -Z), the plane's `z_dir` (its normal) must be world Y
-- not world -Z as first guessed. Getting this backwards silently rotated the whole profile 90 degrees
into the wrong plane; caught by checking the bounding box and probe volumes below, the same class of
`Plane` gotcha that bit the side-cutout work earlier in the project.

In [2]:
def build_slot_block() -> object:
    """Build a rectangular block with one tapered V-slot subtracted from its top."""
    with BuildPart() as block:
        Box(
            BLOCK_WIDTH,
            LANE_LENGTH,
            BLOCK_HEIGHT,
            align=(Align.CENTER, Align.CENTER, Align.MIN),
        )
        top_z = block.part.bounding_box().max.Z

        cut_plane = Plane(origin=(0, 0, top_z), x_dir=(1, 0, 0), z_dir=(0, 1, 0))
        with BuildSketch(cut_plane) as slot_sketch:
            with BuildLine():
                Line((-MOUTH_WIDTH / 2, 0), (MOUTH_WIDTH / 2, 0))
                Line((MOUTH_WIDTH / 2, 0), (APEX_WIDTH / 2, TAPER_DEPTH))
                Line((APEX_WIDTH / 2, TAPER_DEPTH), (APEX_WIDTH / 2, TAPER_DEPTH + RELIEF_DEPTH))
                Line(
                    (APEX_WIDTH / 2, TAPER_DEPTH + RELIEF_DEPTH),
                    (-APEX_WIDTH / 2, TAPER_DEPTH + RELIEF_DEPTH),
                )
                Line((-APEX_WIDTH / 2, TAPER_DEPTH + RELIEF_DEPTH), (-APEX_WIDTH / 2, TAPER_DEPTH))
                Line((-APEX_WIDTH / 2, TAPER_DEPTH), (-MOUTH_WIDTH / 2, 0))
            make_face()
        extrude(to_extrude=slot_sketch.sketch.faces(), amount=LANE_LENGTH, both=True, mode=Mode.SUBTRACT)
    return block.part


block = build_slot_block()
print(f"is_valid(): {block.is_valid()}")
print(f"bounding box: {block.bounding_box()}")

is_valid(): True
bounding box: bbox: -20.0 <= x <= 20.0, -42.0 <= y <= 42.0, 0.0 <= z <= 18.0


## 3. Self-centring: where does a blade of a given thickness wedge?

The V narrows linearly with depth, so solving `width(depth) == thickness` gives the depth at which a
blade's two faces contact the taper walls and stop sinking further. A thicker blade should wedge
*shallower* (smaller depth) than a thinner one.

In [3]:
def slot_width_at_depth(depth: float) -> float:
    """Return the V's width at a given depth below the mouth (0 at the mouth)."""
    if depth <= TAPER_DEPTH:
        fraction = depth / TAPER_DEPTH
        return MOUTH_WIDTH - fraction * (MOUTH_WIDTH - APEX_WIDTH)
    return APEX_WIDTH


def wedge_depth_for_thickness(thickness: float) -> float:
    """Return the depth at which a blade of this thickness wedges in the taper."""
    if not APEX_WIDTH <= thickness <= MOUTH_WIDTH:
        msg = f"thickness {thickness} outside supported range [{APEX_WIDTH}, {MOUTH_WIDTH}]"
        raise ValueError(msg)
    fraction = (MOUTH_WIDTH - thickness) / (MOUTH_WIDTH - APEX_WIDTH)
    return fraction * TAPER_DEPTH


thick = 3.0 * MM
thin = 1.5 * MM
thick_depth = wedge_depth_for_thickness(thick)
thin_depth = wedge_depth_for_thickness(thin)
print(f"thick ({thick} mm) wedges at depth {thick_depth:.3f} mm (shallower is higher)")
print(f"thin  ({thin} mm) wedges at depth {thin_depth:.3f} mm")
assert thick_depth < thin_depth, "thicker blade should wedge shallower than thinner"

thick (3.0 mm) wedges at depth 5.143 mm (shallower is higher)
thin  (1.5 mm) wedges at depth 10.286 mm


## 4. Physically probe the geometry

Confirm the analytic prediction against the actual solid, using the same intersect-a-probe-box
technique used throughout this project's test suite (`_region_volume` in `tests/test_cutlery_bin.py`):
a thin box at exactly a blade's predicted wedge depth should sit in pure void (near-zero intersection
with the block), while an oversized box at the same depth should collide with the taper walls.

In [4]:
def region_volume(solid: object, center: tuple, size: tuple) -> float:
    """Return the volume of `solid` inside an axis-aligned probe box."""
    with BuildPart() as probe:
        with BuildPart(Location(center)):
            Box(*size)
        add(solid, mode=Mode.INTERSECT)
    return probe.part.volume if probe.part else 0.0


top_z = BLOCK_HEIGHT
for label, thickness, depth in (("thick", thick, thick_depth), ("thin", thin, thin_depth)):
    z = top_z - depth
    vol = region_volume(block, (0, 0, z), (thickness * 0.98, 5.0, 0.05))
    print(f"{label} blade proxy at its own wedge depth: intersection volume = {vol:.4f} mm^3 (want ~0)")
    assert vol < 0.01, f"{label} blade proxy unexpectedly collides with solid"

z_thick_depth = top_z - thick_depth
local_width = slot_width_at_depth(thick_depth)
oversized = local_width + 1.0
vol_oversized = region_volume(block, (0, 0, z_thick_depth), (oversized, 5.0, 0.5))
print(
    f"oversized proxy ({oversized:.2f} mm) at thick-blade depth: "
    f"intersection volume = {vol_oversized:.4f} mm^3 (want > 0)"
)
assert vol_oversized > 0.01, "expected an oversized proxy to collide with the taper wall"

thick blade proxy at its own wedge depth: intersection volume = 0.0000 mm^3 (want ~0)
thin blade proxy at its own wedge depth: intersection volume = 0.0000 mm^3 (want ~0)
oversized proxy (4.00 mm) at thick-blade depth: intersection volume = 2.5000 mm^3 (want > 0)


## 5. Apex relief: the cutting edge floats

The relief channel (below `TAPER_DEPTH`) is a constant-width corridor narrower than any supported blade
thickness, so a blade's edge -- which sits below where its faces wedge -- should hang in open space with
no contact, regardless of blade thickness.

In [5]:
relief_mid_z = top_z - (TAPER_DEPTH + RELIEF_DEPTH / 2)
vol_relief = region_volume(block, (0, 0, relief_mid_z), (APEX_WIDTH * 0.98, 5.0, RELIEF_DEPTH * 0.9))
print(f"relief channel interior: intersection volume = {vol_relief:.4f} mm^3 (want ~0)")
assert vol_relief < 0.01, "expected the relief channel to be fully open"

print("\nAll prototype checks passed -- the tapered slot self-centres and the edge floats.")

relief channel interior: intersection volume = 0.0000 mm^3 (want ~0)

All prototype checks passed -- the tapered slot self-centres and the edge floats.


## 6. Chosen defaults for the Prima 7-knife set

These values (mouth 4.5 mm, apex 1.0 mm, taper depth 12 mm, relief depth 3 mm) carry over directly into
`KnifeBlockParameters` in `cutlery_bin.py`. The target set's spines measure 2-3 mm, comfortably inside
`[APEX_WIDTH, MOUTH_WIDTH]`.